In [1]:

# Cell 1: Imports and configuration

import os
import time
import random
import copy
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from torchvision.models import AlexNet_Weights

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Experimental settings
DATA_ROOT = "/content/data"
BATCH_SIZE = 128
NUM_WORKERS = 2

CODE_LENGTHS = [16, 32, 48, 64]

# Continuation schedule: beta gradually increases toward binary sign
BETA_SCHEDULE = [1, 2, 4, 8, 16]
EPOCHS_PER_BETA = 3

LEARNING_RATE = 1e-5
HASH_LAYER_LR_MULTIPLIER = 10
WEIGHT_DECAY = 5e-4
ALPHA = 1.0

# Standard CIFAR-10 split:
# 50,000 training images; 10,000 test queries
EVAL_QUERY_LIMIT = None  # None = evaluate all 10,000 queries

print("Code lengths:", CODE_LENGTHS)
print("Beta schedule:", BETA_SCHEDULE)


Device: cuda
Code lengths: [16, 32, 48, 64]
Beta schedule: [1, 2, 4, 8, 16]


In [ ]:

# Cell 2: CIFAR-10 datasets

imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std  = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=train_transform
)

# Separate evaluation dataset without random augmentation
train_eval_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=eval_transform
)

test_dataset = datasets.CIFAR10(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

class_names = train_dataset.classes

print("Classes:", class_names)
print("Train images:", len(train_dataset))
print("Test queries:", len(test_dataset))

# Visualize sample images in original CIFAR-10 resolution
raw_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=True
)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    image, label = raw_dataset[i]
    ax.imshow(image)
    ax.set_title(class_names[label])
    ax.axis("off")

plt.tight_layout()
plt.show()


 47%|████▋     | 80.0M/170M [29:32<28:51, 52.3kB/s]

In [3]:

# Cell 3: HashNet model based on pretrained AlexNet

class HashNet(nn.Module):
    def __init__(self, code_length=16, pretrained=True):
        super().__init__()

        weights = AlexNet_Weights.DEFAULT if pretrained else None
        self.backbone = models.alexnet(weights=weights)

        # Replace the last classification layer with a hash layer
        in_features = self.backbone.classifier[6].in_features
        self.backbone.classifier[6] = nn.Linear(
            in_features, code_length
        )

        nn.init.normal_(
            self.backbone.classifier[6].weight, mean=0.0, std=0.01
        )
        nn.init.constant_(self.backbone.classifier[6].bias, 0.0)

    def forward(self, x):
        # Continuous hash representation z
        return self.backbone(x)

    def continuation_output(self, x, beta):
        # Smooth approximation to sign(z)
        z = self.forward(x)
        u = torch.tanh(beta * z)
        return z, u

    @torch.no_grad()
    def encode(self, x):
        self.eval()
        z = self.forward(x)
        # Binary code represented as -1/+1
        return torch.where(
            z >= 0,
            torch.ones_like(z),
            -torch.ones_like(z)
        )


# Example: instantiate the 16-bit model
model = HashNet(code_length=16, pretrained=True).to(DEVICE)

print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:03<00:00, 78.3MB/s]


HashNet(
  (backbone): AlexNet(
    (features): Sequential(
      (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
      (1): ReLU(inplace=True)
      (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (4): ReLU(inplace=True)
      (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU(inplace=True)
      (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (9): ReLU(inplace=True)
      (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
    (classifier): Sequential(
      (0): Dropout(p=0.5, inplace=False)
      

In [4]:

# Cell 4: HashNet pairwise objective and training

def pairwise_hashnet_loss(z, labels, beta, alpha=1.0):
    """
    z: [B, K] continuous hash outputs
    labels: [B]
    beta: continuation parameter

    Pairwise logistic objective:
        theta_ij = 0.5 * u_i^T u_j
        u_i = tanh(beta * z_i)

    Similar pairs have target 1; dissimilar pairs target 0.
    """
    u = torch.tanh(beta * z)

    # Pairwise similarity labels
    similarity = (
        labels.unsqueeze(1) == labels.unsqueeze(0)
    ).float()

    # Exclude self-pairs
    batch_size = labels.size(0)
    mask = ~torch.eye(
        batch_size, dtype=torch.bool, device=labels.device
    )

    theta = 0.5 * (u @ u.T)
    logits = alpha * theta

    targets = similarity[mask]
    pair_logits = logits[mask]

    positive = targets == 1
    negative = targets == 0

    # Balance total contribution from positive and negative pairs
    n_pos = positive.sum().clamp(min=1)
    n_neg = negative.sum().clamp(min=1)

    weights = torch.zeros_like(targets)
    weights[positive] = 0.5 / n_pos
    weights[negative] = 0.5 / n_neg

    loss_each = F.binary_cross_entropy_with_logits(
        pair_logits, targets, reduction="none"
    )

    loss = (weights * loss_each).sum()
    return loss


def train_one_stage(model, loader, optimizer, beta, epochs):
    model.train()
    history = []

    for epoch in range(epochs):
        running_loss = 0.0
        count = 0
        start = time.time()

        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            z = model(images)
            loss = pairwise_hashnet_loss(
                z, labels, beta=beta, alpha=ALPHA
            )

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            count += 1

        avg_loss = running_loss / max(count, 1)
        elapsed = time.time() - start
        history.append(avg_loss)

        print(
            f"beta={beta:>2} | epoch={epoch+1}/{epochs} "
            f"| loss={avg_loss:.5f} | time={elapsed:.1f}s"
        )

    return history


def fit_hashnet(code_length):
    model = HashNet(
        code_length=code_length, pretrained=True
    ).to(DEVICE)

    hash_params = list(model.backbone.classifier[6].parameters())
    hash_ids = {id(p) for p in hash_params}
    backbone_params = [
        p for p in model.parameters() if id(p) not in hash_ids
    ]

    optimizer = torch.optim.SGD(
        [
            {"params": backbone_params, "lr": LEARNING_RATE},
            {
                "params": hash_params,
                "lr": LEARNING_RATE * HASH_LAYER_LR_MULTIPLIER
            },
        ],
        momentum=0.9,
        weight_decay=WEIGHT_DECAY
    )

    all_history = []

    for beta in BETA_SCHEDULE:
        print(f"\n--- Continuation stage beta={beta} ---")
        stage_history = train_one_stage(
            model, train_loader, optimizer,
            beta=beta, epochs=EPOCHS_PER_BETA
        )
        all_history.extend(stage_history)

    return model, all_history


In [5]:

# Cell 5: Encode all images and evaluate retrieval

@torch.no_grad()
def extract_binary_codes(model, loader):
    model.eval()
    codes = []
    labels_all = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        z = model(images)

        binary = torch.where(
            z >= 0,
            torch.ones_like(z),
            -torch.ones_like(z)
        )

        codes.append(binary.cpu().to(torch.int8))
        labels_all.append(labels.cpu())

    return torch.cat(codes), torch.cat(labels_all)


def calculate_map(query_codes, query_labels,
                  database_codes, database_labels,
                  top_k=None, query_limit=None):
    """
    Hamming ranking using {-1, +1} binary codes.

    mAP@All:
        top_k=None, ranks all gallery images.

    mAP@K:
        top_k=K, evaluates only the first K retrieved images.
        AP denominator is the number of relevant images in the
        full gallery, a common truncated-AP convention.
    """
    if query_limit is not None:
        query_codes = query_codes[:query_limit]
        query_labels = query_labels[:query_limit]

    q = query_codes.to(torch.int32)
    db = database_codes.to(torch.int32)
    q_labels = query_labels.numpy()
    db_labels = database_labels.numpy()

    n_db = len(db_labels)
    if n_db == 0:
        raise ValueError("Gallery is empty.")

    k_eval = n_db if top_k is None else min(top_k, n_db)

    # With +/-1 codes, Hamming distance = (K - dot product) / 2
    code_length = db.shape[1]
    average_precisions = []

    # Chunk queries to reduce peak memory usage
    chunk_size = 64

    for start in range(0, len(q), chunk_size):
        end = min(start + chunk_size, len(q))

        # Larger dot product means smaller Hamming distance
        similarity = q[start:end] @ db.T

        # Sort by descending similarity
        order = torch.argsort(
            similarity, dim=1, descending=True
        )[:, :k_eval].cpu().numpy()

        for local_idx, ranked_indices in enumerate(order):
            query_idx = start + local_idx
            ranked_labels = db_labels[ranked_indices]

            relevant = (
                ranked_labels == q_labels[query_idx]
            ).astype(np.int32)

            total_relevant = np.sum(
                db_labels == q_labels[query_idx]
            )

            if total_relevant == 0:
                continue

            cumulative_hits = np.cumsum(relevant)
            ranks = np.arange(1, len(relevant) + 1)

            precision_at_rank = cumulative_hits / ranks
            ap = np.sum(precision_at_rank * relevant) / total_relevant

            average_precisions.append(ap)

    return float(np.mean(average_precisions))


def evaluate_hashnet(model, query_limit=None):
    db_codes, db_labels = extract_binary_codes(
        model, train_eval_loader
    )
    q_codes, q_labels = extract_binary_codes(
        model, test_loader
    )

    map_all = calculate_map(
        q_codes, q_labels, db_codes, db_labels,
        top_k=None, query_limit=query_limit
    )

    # CIFAR-10 train gallery has 50,000 images.
    # Therefore, mAP@54000 cannot be separately measured here.
    map_54000 = None
    if len(db_labels) >= 54000:
        map_54000 = calculate_map(
            q_codes, q_labels, db_codes, db_labels,
            top_k=54000, query_limit=query_limit
        )

    return {
        "mAP@All": map_all,
        "mAP@54000": map_54000,
        "num_queries": min(
            len(q_labels),
            query_limit if query_limit is not None else len(q_labels)
        ),
        "gallery_size": len(db_labels),
    }


In [7]:
# Trích xuất mã nhị phân từ tập truy vấn và tập cơ sở dữ liệu
query_codes, query_labels = extract_binary_codes(model, test_loader)
database_codes, database_labels = extract_binary_codes(model, train_eval_loader)

# Chuyển đổi sang định dạng mong muốn trên thiết bị hoạt động
q = query_codes.to(DEVICE, dtype=torch.int32)
db = database_codes.to(DEVICE, dtype=torch.int32)

print("Query shape:", q.shape)
print("Database shape:", db.shape)

KeyboardInterrupt: 

In [ ]:

# Cell 6: Train and evaluate all code lengths

os.makedirs("/content/hashnet_results", exist_ok=True)

results = []
training_histories = {}

for bits in CODE_LENGTHS:
    print("\n" + "=" * 60)
    print(f"Training HashNet with {bits}-bit codes")
    print("=" * 60)

    start_time = time.time()

    model, history = fit_hashnet(bits)
    training_histories[bits] = history

    metrics = evaluate_hashnet(
        model, query_limit=EVAL_QUERY_LIMIT
    )

    elapsed = time.time() - start_time

    row = {
        "Method": "HashNet reimplementation",
        "Bits": bits,
        "mAP@All": metrics["mAP@All"],
        "mAP@54000": metrics["mAP@54000"],
        "Queries": metrics["num_queries"],
        "Gallery": metrics["gallery_size"],
        "Training+Evaluation (seconds)": elapsed,
        "Beta schedule": str(BETA_SCHEDULE),
        "Epochs per beta": EPOCHS_PER_BETA,
    }

    results.append(row)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "bits": bits,
            "history": history,
            "metrics": metrics,
            "beta_schedule": BETA_SCHEDULE,
            "epochs_per_beta": EPOCHS_PER_BETA,
        },
        f"/content/hashnet_results/hashnet_{bits}bit.pth"
    )

    print("Metrics:", metrics)
    print(f"Elapsed: {elapsed / 60:.1f} minutes")

results_df = pd.DataFrame(results)
results_df.to_csv(
    "/content/hashnet_results/experimental_results.csv",
    index=False
)

display(results_df)

print("Saved results to /content/hashnet_results/")



Training HashNet with 16-bit codes

--- Continuation stage beta=1 ---
